In [1]:
import pandas as pd
import numpy as np

## Classification

In [6]:
from sklearn.datasets import load_iris
iris= load_iris(as_frame= True)
df= iris.frame
df= df.sample(len(df), replace= False)
df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
98,5.1,2.5,3.0,1.1,1
118,7.7,2.6,6.9,2.3,2
135,7.7,3.0,6.1,2.3,2
34,4.9,3.1,1.5,0.2,0
88,5.6,3.0,4.1,1.3,1
...,...,...,...,...,...
144,6.7,3.3,5.7,2.5,2
111,6.4,2.7,5.3,1.9,2
49,5.0,3.3,1.4,0.2,0
11,4.8,3.4,1.6,0.2,0


In [7]:
x= df.drop(columns= ['target'])
y= df['target']

In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import cross_val_score

dt= DecisionTreeClassifier()

score= cross_val_score(dt, x, y, cv= 3)
print(np.mean(score).round(2))


0.96


In [18]:
bag= BaggingClassifier(
    estimator= DecisionTreeClassifier(),
    n_estimators= 100,
    max_samples= 0.5,
    max_features= 4,
    bootstrap_features= False,
    bootstrap= True
)
score= cross_val_score(bag, x, y, cv= 3)
print(np.mean(score).round(2))



0.96


### Pasting

In [20]:
bag= BaggingClassifier(
    estimator= DecisionTreeClassifier(),
    n_estimators= 100,
    max_samples= 0.5,
    max_features= 4,
    bootstrap_features= False,
    bootstrap= False            #no repetation during sampling
)
score= cross_val_score(bag, x, y, cv= 3)
print(np.mean(score).round(2))


0.95


### Random Subspace Method

In [25]:
bag= BaggingClassifier(
    estimator= DecisionTreeClassifier(),
    n_estimators= 100,
    max_samples= 1.0,
    max_features= 0.25,
    bootstrap_features= True,    #feature sampling instead of row sampling
    bootstrap= False            
)
score= cross_val_score(bag, x, y, cv= 3)
print(np.mean(score).round(2))   #only do feature sampling if we have many features

0.93


### Random Patches 
    - Combination of both row sampling and column sampling

In [ ]:
bag= BaggingClassifier(
    estimator= DecisionTreeClassifier(),
    n_estimators= 100,
    max_samples= 0.25,
    max_features= 0.5,
    bootstrap_features= True,    
    bootstrap= True            
)
score= cross_val_score(bag, x, y, cv= 3)
print(np.mean(score).round(2))   

0.95


In [28]:
from sklearn.model_selection import  train_test_split,  GridSearchCV

x_train, x_test, y_train, y_test= train_test_split(x, y, random_state= 42)

"""
Always split before knowing the values of hyperparamter. If we donot do split for hyperparam finding then the grdi search cv will use all the data to fins the best param and gives best param. BUt we donot have any unseen data left to know how well it works.
In case of cross val usually no need because we just want to know the accuracy of model or how good the model works"""




'\nAlways split before knowing the values of hyperparamter. If we donot do split for hyperparam finding then the grdi search cv will use all the data to fins the best param and gives best param. BUt we donot have any unseen data left to know how well it works.\nIn case of cross val usually no need because we just want to know the accuracy of model or how good the model works'

In [38]:
param= {
    'n_estimators': [x for x in list(range(50, 501, 50))],
    'max_samples': [ 0.25, 0.5, 0.75, 1.0],
    'max_features': [ 0.25, 0.5, 0.75, 1.0],
    'bootstrap_features': [True, False],
    'bootstrap': [True, False]

}

In [39]:
best= GridSearchCV(
    estimator= BaggingClassifier(estimator= DecisionTreeClassifier()),
    n_jobs= -1,
    scoring= 'accuracy',
    param_grid= param

)

In [40]:
best.fit(x_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",BaggingClassi...eClassifier())
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'bootstrap': [True, False], 'bootstrap_features': [True, False], 'max_features': [0.25, 0.5, ...], 'max_samples': [0.25, 0.5, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for 

In [41]:
best.best_score_

np.float64(0.9652173913043478)

In [42]:
best.best_params_

{'bootstrap': True,
 'bootstrap_features': True,
 'max_features': 0.5,
 'max_samples': 0.25,
 'n_estimators': 200}

Regression is also same as the classification from parameters to types and everything. SO no code here